In [13]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA


from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score


In [14]:
df_dataset = pd.read_parquet("../../data/processed/dataset_features.parquet")

df_dataset = df_dataset[df_dataset["condition"].isin(["EO", "EC"])]

df_dataset["y"] = df_dataset["condition"].map({"EO": 0, "EC": 1})

In [ ]:
df_dataset.columns

Index(['degree_0', 'degree_1', 'degree_2', 'degree_3', 'degree_4', 'degree_5',
       'degree_6', 'degree_7', 'degree_8', 'degree_9',
       ...
       'age_65-70', 'age_70-75', 'age_75-80', 'gender_2', 'handedness_left',
       'handedness_right', 'education_gymnasium', 'education_realschule',
       'relationship_status_yes', 'y'],
      dtype='object', length=410)

: 

In [15]:
NODE_FEATURES = [
    c for c in df_dataset.columns
    if c.startswith(("degree_", "clustering_", "betweenness_", "hub_"))
]

In [16]:
df_dataset = df_dataset.rename(columns={
    "Gender_ 1=female_2=male": "gender",
    "Age": "age",
    "Handedness": "handedness",
    "Education": "education",
    "DRUG_0=negative_1=Positive": "drug",
    "Smoking_num_(Non-smoker=1, Occasional Smoker=2, Smoker=3)": "smoking",
    "SKID_Diagnoses": "skid_diagnoses",
    "SKID_Diagnoses 1": "skid_diagnoses_1",
    "SKID_Diagnoses 2": "skid_diagnoses_2",
    "Comments_SKID_assessment": "comments_skid_assessment",
    "Hamilton_Scale": "hamilton_scale",
    "BSL23_sumscore": "bsl23_sumscore",
    "BSL23_behavior": "bsl23_behavior",
    "AUDIT": "audit",
    "Standard_Alcoholunits_Last_28days": "standard_alcoholunits_last_28days",
    "Alcohol_Dependence_In_1st-3rd_Degree_relative": "alcohol_dependence_in_1st_3rd_degree_relative",
    "Relationship_Status": "relationship_status"
})

In [17]:
META_FEATURES = ["age", "gender"]

In [18]:
def evaluate_model(X, y, model):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "acc": "accuracy",
        "auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring
    )

    acc_mean = scores["test_acc"].mean()
    acc_std = scores["test_acc"].std()

    auc_mean = scores["test_auc"].mean()
    auc_std = scores["test_auc"].std()

    print(
        "=== Model Evaluation (Stratified 5-Fold CV) ===\n"
        f"Mean Accuracy : {acc_mean:.3f} ± {acc_std:.3f}\n"
        f"Mean ROC-AUC  : {auc_mean:.3f} ± {auc_std:.3f}\n"
        "Note: ROC-AUC values close to 0.5 indicate random performance."
    )

    return {
        "accuracy_mean": acc_mean,
        "auc_mean": auc_mean
    }


A  função acima treina e avalia um modelo de classificação usando validação cruzada estratificada com 5 folds, calcula acurácia e AUC em cada fold e retorna a média dessas métricas.

### Regressão Logística

In [19]:
X = df_dataset[NODE_FEATURES].select_dtypes(include="number").dropna(axis=1)
y = df_dataset["y"]

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.435 ± 0.093
Mean ROC-AUC  : 0.425 ± 0.121
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.43506493506493504),
 'auc_mean': np.float64(0.42495867768595036)}

Avaliação do modelo por validação cruzada estratificada (5 folds).
O modelo apresentou acurácia média de 42,1% e AUC média de 0,42, indicando um desempenho inferior ao esperado para um classificador binário.

A acurácia próxima a 50% sugere que o modelo possui baixa capacidade preditiva, enquanto o valor de AUC abaixo de 0,5 indica que o modelo não consegue separar adequadamente as classes, apresentando desempenho próximo — ou até inferior — ao de um classificador aleatório.

Esses resultados indicam a necessidade de revisão das features, engenharia de atributos, ajuste de hiperparâmetros ou a avaliação de modelos mais complexos.

### Regressão Logística com PCA

In [20]:

pipe_pca_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000))
])

evaluate_model(X, y, pipe_pca_lr)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.489 ± 0.076
Mean ROC-AUC  : 0.505 ± 0.101
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.4891774891774891),
 'auc_mean': np.float64(0.5054545454545455)}

Os modelos de classificação supervisionada baseados em regressão logística apresentaram desempenho próximo ao aleatório (AUC ≈ 0,45), mesmo após a aplicação de PCA. Esse resultado sugere que, em uma população saudável, as métricas globais e locais de conectividade funcional não são suficientes para discriminar automaticamente as condições analisadas, reforçando a elevada variabilidade interindividual típica de cérebros saudáveis.

### Regressão Logística com PCA e metadados

In [22]:
meta = df_dataset[META_FEATURES].copy()
meta["gender"] = pd.to_numeric(meta["gender"], errors="coerce")
meta = pd.get_dummies(meta, columns=["age"], drop_first=True)

X_meta = pd.concat([X, meta], axis=1).dropna()
y_meta = df_dataset.loc[X_meta.index, "y"]

pipe_pca_meta = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=10)),
    ("clf", LogisticRegression(max_iter=1000)),
])

evaluate_model(X_meta, y_meta, pipe_pca_meta)


=== Model Evaluation (Stratified 5-Fold CV) ===
Mean Accuracy : 0.519 ± 0.064
Mean ROC-AUC  : 0.528 ± 0.073
Note: ROC-AUC values close to 0.5 indicate random performance.


{'accuracy_mean': np.float64(0.519047619047619),
 'auc_mean': np.float64(0.5279338842975208)}